# Read summary: SEGUL vs fastp and FastQC

Compares `segul read summary` with two widely used read QC tools on the same
gzipped FASTQ files from `data/`:

- **fastp** in report-only mode (`-A -G -Q -L`, no output reads, JSON and HTML
  report).
- **FastQC** restricted to the modules SEGUL Complete also computes, so both
  tools compute the same features.

Results come from `uv run segul2-bench read-summary run` and are read from
`results/read-summary/latest.csv`. Re-run this notebook with:

```bash
uv run jupyter nbconvert --to notebook --execute --inplace notebooks/read_summary.ipynb
```

Two scopes are measured: **file** (each FASTQ file on its own) and
**dataset** (all files of a dataset at once).

In [ ]:
import json
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import colormaps
from matplotlib.colors import to_hex
from matplotlib.ticker import FuncFormatter

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "results" / "read-summary"
FIGURES = ROOT / "figures" / "read-summary"
FIGURES.mkdir(parents=True, exist_ok=True)

runs = pd.read_csv(RESULTS / "latest.csv")
meta = json.loads((RESULTS / "latest.json").read_text())
assert (runs["exit_code"] == 0).all(), "Some runs failed"
THREADS = meta["threads"]

print(f"Date:    {meta['date']}")
print(f"Machine: {meta['machine']['cpu']}, {meta['machine']['logical_cpus']} logical CPUs, "
      f"{meta['machine']['memory_gb']} GB, {meta['machine']['os']}")
print(f"Threads: {THREADS} (segul RAYON_NUM_THREADS, fastp -w, FastQC --threads)")
print(f"Repeats: {meta['repeats']} (+{meta['warmup']} warmup, not recorded)")
print(f"segul:   {meta['segul']['version']} ({meta['segul']['git_rev']}), dup sketch {meta['segul']['dup_size']:,}")
print(f"fastp:   {meta['fastp']['version']}")
print(f"FastQC:  {meta['fastqc']['version']} on {meta['fastqc']['java']}, "
      f"ignored modules: {', '.join(meta['fastqc']['ignored_modules'])}, "
      f"--dup_length {meta['fastqc'].get('dup_length')}")
for d in meta["datasets"]:
    print(f"Dataset {d['id']}: {d['n_files']} files, {d['size_mb']} MB gz. {d['description']}")

### What each tool computes

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.DataFrame(meta["computes"].items(), columns=["tool and mode", "computes"]).style.hide(axis="index")

In [ ]:
# ColorBrewer Dark2, one fixed color per tool. Markers and line styles mark the
# mode, so identity never relies on color alone.
DARK2 = [to_hex(c) for c in colormaps["Dark2"].colors]
TOOL_COLOR = {"segul": DARK2[0], "fastp": DARK2[1], "fastqc": DARK2[2]}
TOOL_NAME = {"segul": "SEGUL", "fastp": "fastp", "fastqc": "FastQC"}
LEVEL = {
    ("segul", "minimal"): "Minimal",
    ("segul", "default"): "Default",
    ("fastp", "report"): "Default",
    ("segul", "complete"): "Complete",
    ("fastp", "complete"): "Complete",
    ("fastqc", "complete"): "Complete",
}
LEVELS = ["Minimal", "Default", "Complete"]
LEVEL_MARKER = {"Minimal": "s", "Default": "o", "Complete": "D"}
LEVEL_LINE = {"Minimal": ":", "Default": "-", "Complete": "--"}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
PLAIN = FuncFormatter(lambda v, _: f"{v:g}")

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 10,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "axes.titleweight": "bold",
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "legend.frameon": False,
})

runs["level"] = [LEVEL[(t, m)] for t, m in zip(runs["tool"], runs["mode"])]
stats = (
    runs.groupby(["dataset", "scope", "input", "input_mb", "n_files", "tool", "mode", "level", "threads"],
                 as_index=False)
    .agg(wall_s=("wall_s", "median"), wall_min=("wall_s", "min"), wall_max=("wall_s", "max"),
         rss_mb=("max_rss_mb", "median"), rss_min=("max_rss_mb", "min"), rss_max=("max_rss_mb", "max"),
         cpu_s=("user_s", "median"), reads=("reads", "max"))
)

### Every tool reads every record

Read counts from SEGUL Default output (last run) against the counts in fastp's
JSON report and FastQC's `fastqc_data.txt`, per file.

In [ ]:
segul_counts = {}
for csv_path in (RESULTS / "work").glob("*/file/*/segul-default/out/*.csv"):
    for _, row in pd.read_csv(csv_path).iterrows():
        segul_counts[row["file_name"].removesuffix(".fq.gz")] = row["read_count"]
files = stats[(stats["scope"] == "file") & stats["tool"].isin(["fastp", "fastqc"])]
check = files[files["mode"].isin(["report", "complete"])].pivot_table(
    index=["dataset", "input"], columns="tool", values="reads", aggfunc="max").astype(int)
check.columns = [f"{c} reads" for c in check.columns]
check["segul reads"] = check.index.get_level_values("input").map(segul_counts)
check["match"] = (check["fastp reads"] == check["segul reads"]) & (check["fastqc reads"] == check["segul reads"])
assert check["match"].all(), "Read counts differ"
check

In [ ]:
def dataset_bars(data, tools, levels, metric, lo, hi, ylabel, title, fname, fmt):
    datasets = list(data["dataset"].unique())
    fig, axes = plt.subplots(1, len(datasets), figsize=(max(3.2, 1.75 * len(levels)) * len(datasets), 3.8),
                             squeeze=False)
    width = 0.8 / len(tools)
    for ax, ds in zip(axes[0], datasets):
        sub = data[data["dataset"] == ds]
        for x, level in enumerate(levels):
            # Center the bars a level has, so tools without that level leave no gap.
            present = [t for t in tools if not sub[(sub["tool"] == t) & (sub["level"] == level)].empty]
            for i, tool in enumerate(present):
                r = sub[(sub["tool"] == tool) & (sub["level"] == level)].iloc[0]
                pos = x + (i - (len(present) - 1) / 2) * width
                ax.bar(pos, r[metric], width - 0.03, color=TOOL_COLOR[tool], label=TOOL_NAME[tool])
                ax.errorbar(pos, r[metric], yerr=[[r[metric] - r[lo]], [r[hi] - r[metric]]],
                            color=INK, linewidth=1, capsize=0)
                ax.annotate(fmt.format(r[metric]), (pos, r[hi]), xytext=(0, 3),
                            textcoords="offset points", ha="center", fontsize=8, color=INK)
        ax.set_title(f"{ds} ({sub['n_files'].iloc[0]} files, {sub['input_mb'].iloc[0] / 1000:.2f} GB gz)",
                     fontsize=10)
        ax.set_xticks(range(len(levels)), levels)
        ax.set_ylabel(ylabel)
        ax.margins(y=0.12)
        ax.grid(axis="x", visible=False)
    handles, labels = axes[0][0].get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    names = [TOOL_NAME[t] for t in tools if TOOL_NAME[t] in unique]
    axes[0][0].legend([unique[n] for n in names], names, loc="upper left")
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    fig.savefig(FIGURES / fname, bbox_inches="tight")
    plt.show()


def file_scaling(data, metric, ylabel, title, fname, label):
    datasets = list(data["dataset"].unique())
    fig, axes = plt.subplots(1, len(datasets), figsize=(5.4 * len(datasets), 4.2),
                             squeeze=False, sharey=True)
    for ax, ds in zip(axes[0], datasets):
        for keys, sub in data[data["dataset"] == ds].groupby(["tool", "mode", "level", "threads"]):
            tool, mode, level, threads = keys
            sub = sub.sort_values("input_mb")
            ax.plot(sub["input_mb"], sub[metric], color=TOOL_COLOR[tool], linestyle=LEVEL_LINE[level],
                    linewidth=2, marker=LEVEL_MARKER[level], markersize=6,
                    markeredgecolor="white", markeredgewidth=1, label=label(tool, mode, threads))
        ax.set_title(ds, fontsize=10)
        ax.set_xlabel("Gzipped FASTQ size (MB)")
        ax.set_yscale("log")
        ax.yaxis.set_major_formatter(PLAIN)
    axes[0][0].set_ylabel(ylabel)
    handles, labels = axes[0][0].get_legend_handles_labels()
    order = sorted(range(len(labels)), key=lambda i: (not labels[i].startswith("SEGUL"), labels[i]))
    fig.legend([handles[i] for i in order], [labels[i] for i in order], ncol=5, fontsize=9,
               loc="lower center", bbox_to_anchor=(0.5, -0.06))
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    fig.savefig(FIGURES / fname, bbox_inches="tight")
    plt.show()


def ratios(data, other):
    wide = data.pivot_table(index=["dataset", "input", "level"], columns="tool",
                            values=["wall_s", "rss_mb", "cpu_s"]).dropna()
    name = TOOL_NAME[other]
    out = pd.DataFrame({
        "SEGUL time (s)": wide[("wall_s", "segul")],
        f"{name} time (s)": wide[("wall_s", other)],
        f"speedup ({name} / SEGUL)": wide[("wall_s", other)] / wide[("wall_s", "segul")],
        "SEGUL CPU (s)": wide[("cpu_s", "segul")],
        f"{name} CPU (s)": wide[("cpu_s", other)],
        "SEGUL peak MB": wide[("rss_mb", "segul")],
        f"{name} peak MB": wide[("rss_mb", other)],
        f"memory ratio ({name} / SEGUL)": wide[("rss_mb", other)] / wide[("rss_mb", "segul")],
    })
    return out.round(2)


def ratio_summary(data, other):
    name = TOOL_NAME[other]
    per_file = ratios(data, other).reset_index()
    cols = [f"speedup ({name} / SEGUL)", f"memory ratio ({name} / SEGUL)"]
    return per_file.groupby(["dataset", "level"])[cols].agg(["median", "min", "max"]).round(1)

## Part 1: SEGUL vs fastp

| Level | SEGUL | fastp |
|---|---|---|
| Minimal | `--mode minimal` | (none: fastp has no count-only mode) |
| Default | `--mode default` | `-A -G -Q -L --dont_eval_duplication` |
| Complete | `--mode complete` | `-A -G -Q -L -p` (duplication + overrepresentation) |

The feature sets overlap but are not identical (see the table above). Both
tools get the same thread count. In the dataset scope SEGUL summarizes all
files in one call; fastp runs once per R1/R2 pair in paired-end mode, one
pair after another (times summed, peak memory = largest pair).

In [ ]:
vs_fastp = stats[stats["tool"].isin(["segul", "fastp"]) & (stats["threads"] == THREADS)]
ds = vs_fastp[vs_fastp["scope"] == "dataset"]
note = f"{THREADS} threads (median, whiskers = min-max)"
dataset_bars(ds, ["segul", "fastp"], LEVELS, "wall_s", "wall_min", "wall_max", "Wall time (s)",
             f"Wall time, whole dataset: SEGUL vs fastp, {note}", "fastp-dataset-time.png", "{:.1f}")
dataset_bars(ds, ["segul", "fastp"], LEVELS, "rss_mb", "rss_min", "rss_max", "Peak memory (MB)",
             f"Peak memory, whole dataset: SEGUL vs fastp, {note}", "fastp-dataset-memory.png", "{:.0f}")

In [ ]:
fp_files = vs_fastp[vs_fastp["scope"] == "file"]
fp_label = lambda tool, mode, threads: f"{TOOL_NAME[tool]} {mode}"
file_scaling(fp_files, "wall_s", "Wall time (s, log)",
             f"Wall time per FASTQ file: SEGUL vs fastp, {THREADS} threads (median of repeats)",
             "fastp-file-time.png", fp_label)
file_scaling(fp_files, "rss_mb", "Peak memory (MB, log)",
             f"Peak memory per FASTQ file: SEGUL vs fastp, {THREADS} threads (median of repeats)",
             "fastp-file-memory.png", fp_label)

In [ ]:
ratios(ds, 'fastp')

In [ ]:
ratio_summary(fp_files, 'fastp')

## Part 2: SEGUL vs FastQC (identical features)

FastQC runs only the modules SEGUL Complete also computes. Per-tile quality
and k-mer content, which SEGUL does not compute, are set to `ignore` in a
FastQC limits file built from FastQC's defaults. FastQC normally truncates
reads longer than 75 bp to 50 bp before finding duplicates; SEGUL compares
full reads, so FastQC runs with `--dup_length 1000` (longer than any read) to
use the same definition. Both tools then compute:

| Feature | SEGUL Complete | FastQC module |
|---|---|---|
| Read count, length range, %GC, encoding | summary CSV | Basic Statistics |
| Per-position quality (mean, P10, quartiles, P90) | `read-pos-summary` | Per base sequence quality |
| Per-position A/C/G/T proportions | `read-pos-summary` | Per base sequence content |
| Per-position N proportion | `read-pos-summary` | Per base N content |
| Per-read mean quality distribution | `read-qscore.csv` | Per sequence quality scores |
| Per-read GC distribution + theoretical curve | `gc.csv` | Per sequence GC content |
| Length distribution | `length.csv` | Sequence Length Distribution |
| Duplication levels (100,000 distinct sequences tracked) | `duplication.csv` | Sequence Duplication Levels |
| Overrepresented sequences | `overrepresented.csv` | Overrepresented sequences |
| Adapter content (Illumina Universal, Nextera, Small RNA, polyA, polyG) | `adapters.csv` | Adapter Content |

Thread budgets match. FastQC uses at most 4 threads per file, so in the
**file** scope both tools get 4 threads. In the **dataset** scope both get the
full thread count and summarize all files in one call.

In [ ]:
vs_fastqc = stats[(stats["level"] == "Complete") & stats["tool"].isin(["segul", "fastqc"])]
qc_ds = vs_fastqc[(vs_fastqc["scope"] == "dataset") & (vs_fastqc["threads"] == THREADS)]
dataset_bars(qc_ds, ["segul", "fastqc"], ["Complete"], "wall_s", "wall_min", "wall_max", "Wall time (s)",
             f"Wall time, whole dataset\nSEGUL vs FastQC, {note}", "fastqc-dataset-time.png", "{:.1f}")
dataset_bars(qc_ds, ["segul", "fastqc"], ["Complete"], "rss_mb", "rss_min", "rss_max", "Peak memory (MB)",
             f"Peak memory, whole dataset\nSEGUL vs FastQC, {note}", "fastqc-dataset-memory.png", "{:.0f}")

In [ ]:
qc_threads = int(vs_fastqc[(vs_fastqc["scope"] == "file") & (vs_fastqc["tool"] == "fastqc")]["threads"].max())
qc_files = vs_fastqc[(vs_fastqc["scope"] == "file") & (vs_fastqc["threads"] == qc_threads)]
qc_label = lambda tool, mode, threads: f"{TOOL_NAME[tool]} ({threads} threads)"
file_scaling(qc_files, "wall_s", "Wall time (s, log)",
             f"Wall time per FASTQ file: SEGUL complete vs FastQC, {qc_threads} threads each (median of repeats)",
             "fastqc-file-time.png", qc_label)
file_scaling(qc_files, "rss_mb", "Peak memory (MB, log)",
             f"Peak memory per FASTQ file: SEGUL complete vs FastQC, {qc_threads} threads each (median of repeats)",
             "fastqc-file-memory.png", qc_label)

In [ ]:
ratios(qc_ds, 'fastqc')

In [ ]:
ratio_summary(qc_files, 'fastqc')

### Do the shared features agree?

Basic statistics and duplication from the last run of each tool, per file.
FastQC reports the percentage of reads left after deduplication; SEGUL
reports the estimated duplicate percentage, so the two should add to about
100. Both estimate duplication from the first 100,000 distinct sequences, so
small differences are expected.

In [ ]:
rows = []
for zpath in sorted((RESULTS / "work").glob("*/file/*/fastqc-complete/*_fastqc.zip")):
    with zipfile.ZipFile(zpath) as z:
        text = z.read(f"{zpath.stem}/fastqc_data.txt").decode()
    fq = {}
    for line in text.splitlines():
        parts = line.split("\t")
        if len(parts) >= 2 and parts[0] in ("Total Sequences", "%GC", "Sequence length"):
            fq[parts[0]] = parts[1]
        if line.startswith("#Total Deduplicated Percentage"):
            fq["dedup"] = float(parts[1])
    segul_csv = next(zpath.parent.parent.glob("segul-complete/out/*-read-summary.csv"))
    s = pd.read_csv(segul_csv).iloc[0]
    rows.append({
        "dataset": zpath.parts[-5], "input": zpath.parts[-3],
        "FastQC reads": int(fq["Total Sequences"]), "SEGUL reads": int(s["read_count"]),
        "FastQC %GC": int(fq["%GC"]), "SEGUL %GC": round(100 * s["GC_content"], 1),
        "FastQC length": fq["Sequence length"], "SEGUL length": f"{s['min_read_length']}-{s['max_read_length']}",
        "FastQC dedup %": round(fq["dedup"], 1), "SEGUL duplicate %": round(s["duplicate_pct_est"], 1),
    })
agree = pd.DataFrame(rows)
agree["dedup + duplicate"] = (agree["FastQC dedup %"] + agree["SEGUL duplicate %"]).round(1)
agree

## Part 3: All three tools side by side

The Complete level is the only one all three tools share, and the dataset
scope at the full thread count is the only setting all three were run with,
so this is the one place they can be put on the same axis. Each label
gives the median and how many times SEGUL's value it is. Whiskers are
min-max over repeats.

In [ ]:
TOOLS = ["segul", "fastqc", "fastp"]
all3 = stats[(stats["scope"] == "dataset") & (stats["threads"] == THREADS) & (stats["level"] == "Complete")]
datasets = list(all3["dataset"].unique())
metrics = [("wall_s", "wall_min", "wall_max", "Wall time (s)", "{:.1f} s"),
           ("rss_mb", "rss_min", "rss_max", "Peak memory (MB)", "{:,.0f} MB")]

fig, axes = plt.subplots(len(metrics), len(datasets), figsize=(5.6 * len(datasets), 4.6), squeeze=False)
for row, (metric, lo, hi, xlabel, fmt) in enumerate(metrics):
    for col, ds in enumerate(datasets):
        ax = axes[row][col]
        sub = all3[all3["dataset"] == ds].set_index("tool").loc[TOOLS]
        base = sub.loc["segul", metric]
        y = range(len(TOOLS))
        ax.barh(y, sub[metric], height=0.6, color=[TOOL_COLOR[t] for t in TOOLS])
        ax.errorbar(sub[metric], y, xerr=[sub[metric] - sub[lo], sub[hi] - sub[metric]],
                    fmt="none", ecolor=INK, elinewidth=1, capsize=0)
        for i, tool in enumerate(TOOLS):
            v = sub.loc[tool, metric]
            text = fmt.format(v) if tool == "segul" else f"{fmt.format(v)}  ({v / base:.0f}×)"
            ax.annotate(text, (sub.loc[tool, hi], i), xytext=(4, 0), textcoords="offset points",
                        va="center", fontsize=8.5, color=INK)
        ax.set_yticks(list(y), [TOOL_NAME[t] for t in TOOLS])
        ax.invert_yaxis()
        ax.set_xlim(0, sub[hi].max() * 1.45)
        ax.grid(axis="y", visible=False)
        ax.set_xlabel(xlabel)
        if row == 0:
            ax.set_title(f"{ds} ({sub['n_files'].iloc[0]} files, {sub['input_mb'].iloc[0] / 1000:.2f} GB gz)",
                         fontsize=10)
fig.suptitle(f"SEGUL vs FastQC vs fastp, Complete level, whole dataset, {THREADS} threads (median)",
             fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES / "all-tools-dataset.png", bbox_inches="tight")
plt.show()

All three tools across levels, whole dataset. SEGUL Minimal has no
counterpart, and FastQC has only the Complete level.

In [ ]:
lv = stats[(stats["scope"] == "dataset") & (stats["threads"] == THREADS)]
dataset_bars(lv, TOOLS, LEVELS, "wall_s", "wall_min", "wall_max", "Wall time (s)",
             f"Wall time, whole dataset: all tools, {note}", "all-tools-levels-time.png", "{:.1f}")
dataset_bars(lv, TOOLS, LEVELS, "rss_mb", "rss_min", "rss_max", "Peak memory (MB)",
             f"Peak memory, whole dataset: all tools, {note}", "all-tools-levels-memory.png", "{:.0f}")

In [ ]:
table = all3.pivot_table(index="dataset", columns="tool", values=["wall_s", "rss_mb"])
summary = pd.DataFrame({
    f"{TOOL_NAME[t]} time (s)": table[("wall_s", t)] for t in TOOLS
} | {
    f"{TOOL_NAME[t]} peak MB": table[("rss_mb", t)] for t in TOOLS
} | {
    f"{TOOL_NAME[t]} / SEGUL time": table[("wall_s", t)] / table[("wall_s", "segul")] for t in TOOLS[1:]
} | {
    f"{TOOL_NAME[t]} / SEGUL memory": table[("rss_mb", t)] / table[("rss_mb", "segul")] for t in TOOLS[1:]
})
summary.round(1)

### Notes on fairness

- All tools read the same gzipped files from a warm page cache (one
  unrecorded warmup run per command) and get the same thread budget. fastp
  starts a reader and a writer thread beyond `-w`, and FastQC runs on the JVM,
  whose garbage collector and JIT threads are not counted in `--threads`.
- FastQC peak memory includes the JVM heap (`-Xmx` is 512 MB per file processed
  in parallel, set by the FastQC launcher).
- fastp runs with trimming and filtering disabled and writes no reads, but it
  still computes its full report (quality and base content curves, k-mers, and
  insert size for pairs). SEGUL Default computes fewer per-position curves than
  fastp report, so the Default comparison slightly favors SEGUL in work done;
  SEGUL Complete computes more than fastp complete (adapter content, full length
  and GC distributions).
- fastp's duplication evaluation allocates a fixed table (about 1 GB at the
  default accuracy level), which dominates its memory in the Complete level
  regardless of file size. SEGUL estimates duplication with a fixed-size sketch
  (100,000 sequences), the same approach as FastQC.
- The two datasets are compressed differently. `raw-reads` files are BGZF
  (block gzip), which SEGUL decompresses in parallel, so it reads them faster
  per MB. `read-tests` files were written by fastp as one gzip stream, which
  must be decompressed serially; that is the bottleneck for SEGUL there, and
  why Minimal is barely faster than Default on that dataset.